In [8]:

from typing import List, Dict, Optional, Any
from pprint import pprint

# Import googletrans artifacts
from googletrans import Translator, LANGUAGES

def is_valid_lang_code(code: Optional[str]) -> bool:
    if not code:
        return True  # None means auto-detect (valid)
    return code in LANGUAGES

def lang_name(code: Optional[str]) -> str:
    if code is None:
        return "auto"
    return LANGUAGES.get(code, "Unknown")


In [9]:

# Create translator and translate a single sentence from English to French
translator = Translator()

sample_text = "Hello, how are you?"
result = translator.translate(sample_text, src="en", dest="fr")
print(f"{result.origin} -> {result.text}")


Hello, how are you? -> Bonjour comment allez-vous?


In [10]:

def batch_translate(texts: List[str], src: Optional[str] = "en", dest: str = "fr") -> List[Dict[str, Any]]:
    """Translate a list of texts. Each translation is isolated with try/except.

    Args:
        texts: List of strings to translate.
        src: Source language code (default 'en'). If None, auto-detect.
        dest: Destination language code (default 'fr').

    Returns:
        List of dicts with keys:
            - original
            - translated (or None on error)
            - src (detected or provided)
            - dest
            - error (optional)
    """
    t = Translator()
    results = []

    if not is_valid_lang_code(src):
        raise ValueError(f"Invalid source language code: {src}")
    if not is_valid_lang_code(dest):
        raise ValueError(f"Invalid destination language code: {dest}")

    for text in texts:
        try:
            if text is None or not str(text).strip():
                raise ValueError("Empty text provided")

            translated = t.translate(text, src=src if src else None, dest=dest)
            results.append({
                "original": text,
                "translated": translated.text,
                "src": translated.src,
                "src_lang": lang_name(translated.src),
                "dest": dest,
                "dest_lang": lang_name(dest)
            })
        except Exception as e:
            results.append({
                "original": text,
                "translated": None,
                "src": src if src else "auto",
                "src_lang": lang_name(src),
                "dest": dest,
                "dest_lang": lang_name(dest),
                "error": str(e)
            })
    return results

# Demo
texts_demo = ["Hello, how are you?", "This will translate fine.", ""]
pprint(batch_translate(texts_demo, src="en", dest="fr"))


[{'dest': 'fr',
  'dest_lang': 'french',
  'original': 'Hello, how are you?',
  'src': 'en',
  'src_lang': 'english',
  'translated': 'Bonjour comment allez-vous?'},
 {'dest': 'fr',
  'dest_lang': 'french',
  'original': 'This will translate fine.',
  'src': 'en',
  'src_lang': 'english',
  'translated': 'Cela se traduira très bien.'},
 {'dest': 'fr',
  'dest_lang': 'french',
  'error': 'Empty text provided',
  'original': '',
  'src': 'en',
  'src_lang': 'english',
  'translated': None}]


In [11]:

t = Translator()
text = "Guten Tag, wie geht es dir?"
detection = t.detect(text)
print(f"Detected language: {detection.lang} ({LANGUAGES.get(detection.lang, 'Unknown')})")


Detected language: de (german)


In [12]:

class LanguageProcessor:
    """Complete language processing system with detection, single, and batch translation."""
    def __init__(self):
        self.translator = Translator()
        self.languages = LANGUAGES

    def detect_language(self, text: str) -> Dict[str, Any]:
        """Detect the language of the given text with error handling."""
        try:
            if text is None or not str(text).strip():
                raise ValueError("Empty text provided")
            detection = self.translator.detect(text)
            return {
                "language_code": detection.lang,
                "language_name": self.languages.get(detection.lang, "Unknown"),
                "confidence": getattr(detection, "confidence", None)
            }
        except Exception as e:
            return {"error": str(e)}

    def translate_text(self, text: str, dest: str = "en", src: Optional[str] = None) -> Dict[str, Any]:
        """Translate a single text with robust error handling and auto-detect when src=None."""
        try:
            if text is None or not str(text).strip():
                raise ValueError("Empty text provided")
            if not is_valid_lang_code(dest):
                raise ValueError(f"Invalid destination language code: {dest}")
            if src is not None and not is_valid_lang_code(src):
                raise ValueError(f"Invalid source language code: {src}")

            translated = self.translator.translate(text, src=src if src else None, dest=dest)
            return {
                "original": text,
                "translated": translated.text,
                "src": translated.src,
                "src_lang": self.languages.get(translated.src, "Unknown"),
                "dest": dest,
                "dest_lang": self.languages.get(dest, "Unknown")
            }
        except Exception as e:
            return {"original": text, "error": str(e)}

    def batch_translate(self, texts: List[str], dest: str = "en", src: Optional[str] = None) -> List[Dict[str, Any]]:
        """Translate multiple texts at once with isolated error handling per text."""
        results = []
        for text in texts:
            results.append(self.translate_text(text, dest=dest, src=src))
        return results


In [13]:

processor = LanguageProcessor()

# 1) Basic translation between different language pairs
print("\n#1 Basic translations")
pprint(processor.translate_text("Hello, how are you?", dest="fr"))
pprint(processor.translate_text("Bonjour, comment allez-vous?", dest="en"))
pprint(processor.translate_text("Hola, ¿cómo estás?", dest="de"))
pprint(processor.translate_text("Guten Tag, wie geht es dir?", dest="es"))

# 2) Multiple texts in different source languages (auto-detect)
print("\n#2 Batch translations with auto-detect")
texts = [
    "Hello, how are you?",
    "Bonjour, comment allez-vous?",
    "Hola, ¿cómo estás?",
    "Guten Tag, wie geht es dir?"
]
pprint(processor.batch_translate(texts, dest="en", src=None))

# 3) Error cases
print("\n#3 Error cases")
# 3a) Invalid language codes
pprint(processor.translate_text("Hello", dest="xx"))
# 3b) Empty text
pprint(processor.translate_text("", dest="fr"))
# 3c) Very long text (1000+ characters)
very_long = "a" * 500
pprint(processor.translate_text(very_long, dest="fr"))



#1 Basic translations
{'error': "'NoneType' object has no attribute 'lower'",
 'original': 'Hello, how are you?'}
{'error': "'NoneType' object has no attribute 'lower'",
 'original': 'Bonjour, comment allez-vous?'}
{'error': "'NoneType' object has no attribute 'lower'",
 'original': 'Hola, ¿cómo estás?'}
{'error': "'NoneType' object has no attribute 'lower'",
 'original': 'Guten Tag, wie geht es dir?'}

#2 Batch translations with auto-detect
[{'error': "'NoneType' object has no attribute 'lower'",
  'original': 'Hello, how are you?'},
 {'error': "'NoneType' object has no attribute 'lower'",
  'original': 'Bonjour, comment allez-vous?'},
 {'error': "'NoneType' object has no attribute 'lower'",
  'original': 'Hola, ¿cómo estás?'},
 {'error': "'NoneType' object has no attribute 'lower'",
  'original': 'Guten Tag, wie geht es dir?'}]

#3 Error cases
{'error': 'Invalid destination language code: xx', 'original': 'Hello'}
{'error': 'Empty text provided', 'original': ''}
{'error': "'NoneType

In [14]:

# Show a few sample language codes to names
sample_keys = list(LANGUAGES.keys())[:15]
print({k: LANGUAGES[k] for k in sample_keys})


{'af': 'afrikaans', 'sq': 'albanian', 'am': 'amharic', 'ar': 'arabic', 'hy': 'armenian', 'az': 'azerbaijani', 'eu': 'basque', 'be': 'belarusian', 'bn': 'bengali', 'bs': 'bosnian', 'bg': 'bulgarian', 'ca': 'catalan', 'ceb': 'cebuano', 'ny': 'chichewa', 'zh-cn': 'chinese (simplified)'}
